Toy Example using haptools simgenotype
==================================================
Generates simulated admixed genotype data with ground-truth local ancestry
labels (breakpoints) using haptools simgenotype.

haptools outputs:
  - <scenario>.vcf.gz        — phased genotypes for simulated admixed individuals
  - <scenario>.bp            — ground-truth ancestry breakpoints (the "true labels")

Prior to running this script:
-------------
  pip install haptools

  This script assumes you have:
    1. A phased reference VCF (chr21 region from 1000G/HGDP data)
    2. The HapMap GRCh38 genetic map for chr21
    3. A sample_info .tsv mapping sample IDs -> population codes

  See INPUTS section below to set these paths before running.

Scenarios
---------
    Trivial          — 100% YRI (AFR) as an unadmixed control
    Two ancestries   — AFR + EUR admixture, 7 generations
    Three ancestries — AFR + EUR + AMR admixture, 7 generations

Usage
-----
  python construct_toy_haptools.py

  Outputs written to ./toy_examples_haptools/

In [5]:
import subprocess
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
# INPUTS 
# ─────────────────────────────────────────────────────────────────────────────

REF_VCF = "1000G_chr21_pruned_renamed.vcf.gz"

# Directory containing HapMap GRCh38 .map files
# e.g.:
#   plink.chr21.GRCh38.map

MAP_DIR = "./"


SAMPLE_INFO = "1000genomes_sampleinfo_pruned.tsv"

#REGION = "chr21:10000000-15000000"
#REGION = "21:10000000-11500000"

REGION   = "21:1-32000000"
#48100155 is max


N_SAMPLES = 1


OUT_DIR = Path("toy_examples_haptools")

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Model file construction
# ─────────────────────────────────────────────────────────────────────────────
# The .dat format (inherited from admix-simu, used directly by haptools):
#
#   Line 1 (header):  <n_haplotypes>  Admixed  <POP1>  <POP2>  ...
#   Line 2+:          <generation>    <p_admixed>  <p_POP1>  <p_POP2>  ...
#
# - n_haplotypes = N_SAMPLES * 2  (diploid)
# - p values on each line must sum to 1.0
# - generation 1 = the founding admixture event
# - p_admixed on line 1 must be 0 (no admixed ancestors yet in gen 1)
# - subsequent lines draw from admixed pool to model ongoing admixture
#
# Population codes must exactly match what is in SAMPLE_INFO.
# ─────────────────────────────────────────────────────────────────────────────

def write_model(path: Path, lines: list[str]):
    """Write a .dat model file from a list of tab-separated row strings."""
    path.write_text("\n".join(lines) + "\n")
    print(f"    model file : {path}")


def build_model_files(model_dir: Path, n_haplotypes: int):
    """Write .dat model files for each scenario."""
    model_dir.mkdir(parents=True, exist_ok=True)

    # ── S1: Unadmixed control — 100% YRI for 1 generation ──
    # All haplotypes draw from YRI. p_admixed=0, p_YRI=1.0.
    # This produces a flat karyogram — everything should be called AFR.
    write_model(model_dir / "S1_trivial.dat", [
        f"{n_haplotypes}\tAdmixed\tYRI",
        "1\t0\t1.0",
    ])
    
    write_model(model_dir / "S1_trivial.dat", [
    f"{n_haplotypes}\tAdmixed\tYRI\tIBS",
    "1\t0\t1.0\t0.0",
    ])

    # ── S2: Two ancestries — AFR + EUR pulse, 7 generations ──
    # Generation 1: 80% YRI (AFR), 20% IBS (EUR).
    # Generations 2-7: purely admixed (draws from admixed pool only).
    # 7 generations ≈ ~175-200 years, appropriate for Americas admixture.
    write_model(model_dir / "S2_two_ancestries.dat", [
        f"{n_haplotypes}\tAdmixed\tYRI\tIBS",
        "1\t0\t0.8\t0.2",
        *[f"{g}\t1.0\t0\t0" for g in range(2, 8)],
    ])

    # ── S3: Three ancestries — AFR + EUR + AMR, 7 generations ──
    # Generation 1: 40% YRI (AFR), 40% IBS (EUR), 20% PEL (AMR proxy).
    # Generations 2-7: purely admixed.
    # PEL (Peruvians) used as AMR reference due to high indigenous AMR ancestry.
    write_model(model_dir / "S3_three_ancestries.dat", [
        f"{n_haplotypes}\tAdmixed\tYRI\tIBS\tPEL",
        "1\t0\t0.4\t0.4\t0.2",
        *[f"{g}\t1.0\t0\t0\t0" for g in range(2, 8)],
    ])




In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Run haptools simgenotype
# ─────────────────────────────────────────────────────────────────────────────

def run_simgenotype(scenario_name: str, model_file: Path, out_prefix: Path):
    """
    Run haptools simgenotype for one scenario.

    Outputs:
      <out_prefix>.vcf.gz   — simulated phased genotypes
      <out_prefix>.bp       — ground-truth ancestry breakpoints
    """
    # haptools expects the --out argument to be the .vcf.gz path
    out_vcf = out_prefix.parent / (out_prefix.name + ".vcf.gz")

    cmd = [
        "haptools", "simgenotype",
        "--model",       str(model_file),
        "--mapdir",      MAP_DIR,
        "--ref_vcf",     REF_VCF,
        "--sample_info", SAMPLE_INFO,
        "--region",      REGION,
        "--out",         str(out_vcf),
    ]

    print(f"\n  Running {scenario_name}...")
    print(f"    cmd: {' '.join(cmd)}")

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"  ERROR in {scenario_name}:")
        print(result.stderr)
    else:
        bp_file = out_prefix.parent / (out_prefix.name + ".bp")
        print(f"    genotypes   → {out_vcf}")
        print(f"    breakpoints → {bp_file}")
        if result.stderr:
            # haptools logs to stderr by default; show last few lines
            for line in result.stderr.strip().splitlines()[-5:]:
                print(f"    [log] {line}")

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# Parse breakpoints into a readable summary (optional inspection utility)
# ─────────────────────────────────────────────────────────────────────────────

def summarize_breakpoints(bp_file: Path):
    if not bp_file.exists():
        print(f"  [skip] {bp_file} not found")
        return

    print(f"\n  Breakpoints summary: {bp_file.name}")
    samples_seen = set()
    n_segments = 0
    pop_counts: dict = {}
    current_sample = None

    with open(bp_file) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split("\t")
            if len(parts) == 1:
                # sample/haplotype header line e.g. "Sample_1_1"
                current_sample = parts[0].rsplit("_", 1)[0]  # "Sample_1"
                samples_seen.add(current_sample)
            elif len(parts) == 4:
                # data line: population  chrom  end_pos  genetic_dist
                pop = parts[0]
                n_segments += 1
                pop_counts[pop] = pop_counts.get(pop, 0) + 1

    print(f"    individuals        : {len(samples_seen)}")
    print(f"    total segments     : {n_segments}")
    print(f"    ancestry seg counts: {pop_counts}")


if __name__ == "__main__":

    n_haplotypes = N_SAMPLES * 2  # diploid

    OUT_DIR.mkdir(exist_ok=True)
    model_dir = OUT_DIR / "models"

    print("\nStep 1: Writing model files...")
    build_model_files(model_dir, n_haplotypes)

    print("\nStep 2: Running haptools simgenotype for each scenario...")

    scenarios = [
        ("S1_trivial",          model_dir / "S1_trivial.dat"),
        ("S2_two_ancestries",   model_dir / "S2_two_ancestries.dat"),
        ("S3_three_ancestries", model_dir / "S3_three_ancestries.dat"),
    ]

    for name, model_file in scenarios:
        out_prefix = OUT_DIR / name
        run_simgenotype(name, model_file, out_prefix)

    print("\nStep 3: Breakpoint summaries (ground truth inspection)...")
    for name, _ in scenarios:
        bp_file = OUT_DIR / (name + ".bp")
        summarize_breakpoints(bp_file)

    print(f"""
Done. Outputs in ./{OUT_DIR}/

For each scenario you now have:
  <name>.vcf.gz   — phased genotypes to feed into your HMM/FLARE as test input
  <name>.bp       — ground-truth local ancestry breakpoints for evaluation

To load breakpoints in Python for evaluation against HMM output:

    import pandas as pd
    bp = pd.read_csv("toy_examples_haptools/S3_three_ancestries.bp",
                     sep="\\t", comment="#",
                     names=["sample","hap","population","chrom","start","end"])

Visualizing tracts (optional):
    haptools karyogram toy_examples_haptools/S3_three_ancestries.bp \\
        --output S3_karyogram.png
""")



Step 1: Writing model files...
    model file : toy_examples_haptools/models/S1_trivial.dat
    model file : toy_examples_haptools/models/S1_trivial.dat
    model file : toy_examples_haptools/models/S2_two_ancestries.dat
    model file : toy_examples_haptools/models/S3_three_ancestries.dat

Step 2: Running haptools simgenotype for each scenario...

  Running S1_trivial...
    cmd: haptools simgenotype --model toy_examples_haptools/models/S1_trivial.dat --mapdir ./ --ref_vcf 1000G_chr21_pruned_renamed.vcf.gz --sample_info 1000genomes_sampleinfo_pruned.tsv --region 21:1-32000000 --out toy_examples_haptools/S1_trivial.vcf.gz
    genotypes   → toy_examples_haptools/S1_trivial.vcf.gz
    breakpoints → toy_examples_haptools/S1_trivial.bp
    [log] [ WARNING] The max_variants parameter was not specified. We have no choice but to append to an ever-growing array, which can lead to memory overuse! (genotypes.py:165)
    [log] [    INFO] Loading genotypes from 2504 samples (genotypes.py:287)
   

### Running against FLARE

In [9]:
import pandas as pd

panel = pd.read_csv("1000genomes_sampleinfo_pruned.tsv", sep="\t", header=None, names=["sample", "pop"])

# Take 10 samples per population
subset = (panel[panel["pop"].isin(["YRI", "IBS", "PEL"])]
          .groupby("pop")
          .head(10))

subset.to_csv("1000genomes_sampleinfo_subset.tsv", sep="\t", header=False, index=False)
print(subset["pop"].value_counts())

# Write sample list for bcftools
subset["sample"].to_csv("samples_subset.txt", index=False, header=False)

IBS    10
PEL    10
YRI    10
Name: pop, dtype: int64


In [11]:
"""
run_flare.py
Runs FLARE local ancestry inference on each toy example scenario.
Outputs go to flare_output/<scenario>/
"""

import subprocess
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
# INPUTS
# ─────────────────────────────────────────────────────────────────────────────

FLARE_JAR    = "flare.jar"   
REF_VCF = "1000G_chr21_subset.vcf.gz"
REF_MAP = "1000genomes_sampleinfo_subset.tsv"
# REF_VCF      = "1000G_chr21_pruned_renamed.vcf.gz"
# REF_MAP      = "1000genomes_sampleinfo_subset.tsv"  # 2-col: sample_id  population
# change the map to have fewer
# REF_VCF = "1000G_chr21_subset.vcf.gz"
# REF_MAP = "1000genomes_sampleinfo_subset.tsv"
CHROM        = "21"
OUT_DIR      = Path("flare_output")

TOY_DIR      = Path("toy_examples_haptools")
SCENARIOS    = ["S1_trivial", "S2_two_ancestries", "S3_three_ancestries"]

# ─────────────────────────────────────────────────────────────────────────────

def run_flare(scenario: str):
    target_vcf = TOY_DIR / f"{scenario}.vcf.gz"
    out_prefix = OUT_DIR / scenario / scenario

    out_prefix.parent.mkdir(parents=True, exist_ok=True)

#     cmd = [
#         "java", "-jar", FLARE_JAR,
#         f"ref={REF_VCF}",
#         f"ref-panel={REF_MAP}",
#         f"gt={target_vcf}",
#         f"map=plink.chr21.GRCh38_renamed.map",  # same genetic map used for simulation
#         f"chrom={CHROM}",
#         f"out={out_prefix}",
#         "em=true",      # run EM to estimate ancestry fractions (recommended)
#         "nthreads=4",
#     ]

#     cmd = [
#     "java", "-jar", FLARE_JAR,
#     f"ref={REF_VCF}",
#     f"ref-panel={REF_MAP}",
#     f"gt={target_vcf}",
#     f"map=plink.chr21.GRCh38_renamed.map",
#     f"out={out_prefix}",
#     "em=true",
#     "nthreads=4",
#     ]
    cmd = [
    "java", "-Xmx8g", "-jar", FLARE_JAR,
    f"ref={REF_VCF}",
    f"ref-panel={REF_MAP}",
    f"gt={target_vcf}",
    f"map=plink.chr21.GRCh38_renamed.map",
    #f"chrom={CHROM}",          # <-- add this back
    f"out={out_prefix}",
    "em=true",
    "em-its=5",                # <-- cap EM iterations
    "min-mac=1",
    "nthreads=4",
    ]
    
    print(f"\nRunning FLARE on {scenario}...")
    print(f"  cmd: {' '.join(cmd)}")

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"  ERROR:\n{result.stderr}")
    else:
        print(f"  Output → {out_prefix}.anc.vcf.gz")
        if result.stderr:
            for line in result.stderr.strip().splitlines()[-5:]:
                print(f"  [log] {line}")


if __name__ == "__main__":
    OUT_DIR.mkdir(exist_ok=True)
    for scenario in SCENARIOS:
        run_flare(scenario)

    print(f"""
Done. For each scenario FLARE outputs:
  <scenario>.anc.vcf.gz   — per-SNP ancestry probabilities for each haplotype

To load and compare against ground truth breakpoints:

    import pandas as pd
    from cyvcf2 import VCF

    # Ground truth
    bp = pd.read_csv("toy_examples_haptools/S2_two_ancestries.bp", sep="\\t",
                     comment="#", header=None)

    # FLARE output — ancestry calls are in the FORMAT field (e.g. ANP tag)
    for variant in VCF("flare_output/S2_two_ancestries/S2_two_ancestries.anc.vcf.gz"):
        print(variant.POS, variant.format("ANP"))
""")


Running FLARE on S1_trivial...
  cmd: java -Xmx8g -jar flare.jar ref=1000G_chr21_subset.vcf.gz ref-panel=1000genomes_sampleinfo_subset.tsv gt=toy_examples_haptools/S1_trivial.vcf.gz map=plink.chr21.GRCh38_renamed.map out=flare_output/S1_trivial/S1_trivial em=true em-its=5 min-mac=1 nthreads=4
  ERROR:


Error     :  File does not exist
Filename  :  1000G_chr21_subset.vcf.gz

java.lang.Throwable: File does not exist
	at blbutil.Validate.getFile(Validate.java:147)
	at admix.AdmixPar.<init>(AdmixPar.java:126)
	at admix.AdmixMain.getPar(AdmixMain.java:179)
	at admix.AdmixMain.main(AdmixMain.java:56)

Terminating program.


Running FLARE on S2_two_ancestries...
  cmd: java -Xmx8g -jar flare.jar ref=1000G_chr21_subset.vcf.gz ref-panel=1000genomes_sampleinfo_subset.tsv gt=toy_examples_haptools/S2_two_ancestries.vcf.gz map=plink.chr21.GRCh38_renamed.map out=flare_output/S2_two_ancestries/S2_two_ancestries em=true em-its=5 min-mac=1 nthreads=4
  ERROR:


Error     :  File does not exist
Filena